In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import datagen as gen

pd.set_option("display.max_rows",100)
pd.set_option("display.max_columns",100)

## Combine the data from all quarters from 2017 to 2019

In [ ]:
gen.load_train_data()


../common_training_data/PUDF_base1_1q2017_cleaned_train.csv
../common_training_data/PUDF_base1_2q2017_cleaned_train.csv
../common_training_data/PUDF_base1_3q2017_cleaned_train.csv
../common_training_data/PUDF_base1_4q2017_cleaned_train.csv
../common_training_data/PUDF_base1_1q2018_cleaned_train.csv
../common_training_data/PUDF_base1_2q2018_cleaned_train.csv
../common_training_data/PUDF_base1_3q2018_cleaned_train.csv
../common_training_data/PUDF_base1_4q2018_cleaned_train.csv
../common_training_data/PUDF_base1_1q2019_cleaned_train.csv
../common_training_data/PUDF_base1_2q2019_cleaned_train.csv
../common_training_data/PUDF_base1_3q2019_cleaned_train.csv
../common_training_data/PUDF_base1_4q2019_cleaned_train.csv


## Create pipeline for preprocessing and training

In [ ]:
categorical_cols = gen.cat_features

Split data for training and testing

In [ ]:
# Import Metrics
from sklearn.metrics import mean_absolute_error 
from sklearn.metrics import root_mean_squared_error 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

TRAIN_FLAG = True

if not TRAIN_FLAG:
# df_small = df.sample(n=2000000, random_state=42)
    df_small = df

    y = df_small["LENGTH_OF_STAY"]
    X = df_small.drop(columns=["LENGTH_OF_STAY","PROLONGED","STRATA"])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify = df_small['STRATA']
    )
else:
    X_train = df.drop(columns=["LENGTH_OF_STAY","PROLONGED","STRATA"])
    y_train = df["LENGTH_OF_STAY"]

In [9]:
# Create the StratifiedKFold
strata_train = df.loc[X_train.index, "STRATA"]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2222
)

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
# from sklearn.ensemble import RandomForestRegressor
# from cuml.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ],
    remainder="passthrough"  # Keeps the 'age' column untouched
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    # ('rf_gpu', RandomForestRegressor(n_estimators=100))
    ("regressor", LinearRegression(n_jobs=64)) # Use linear regression
])


## Data training

In [11]:
import numpy as np
from scipy.special import huber
from sklearn.metrics import make_scorer, mean_pinball_loss # For Quantile Loss

def huber_loss(y_true, y_pred, delta=2.0):
    error = y_true - y_pred
    is_small = np.abs(error) <= delta

    squared = 0.5 * error**2
    linear = delta * (np.abs(error) - 0.5 * delta)

    return np.mean(np.where(is_small, squared, linear))

huber_scorer = make_scorer(
    huber_loss,
    delta=2.0,
    greater_is_better=False)

In [12]:
mae_scores = []
rmse_scores = []
quantile_scores = []
huber_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    model_pipeline.fit(X_tr, y_tr)
    pred_y = model_pipeline.predict(X_val)

    mae_score = mean_absolute_error(y_val, pred_y)
    rmse_score = root_mean_squared_error(y_val, pred_y)

    mae_scores.append(mae_score)
    rmse_scores.append(rmse_score)

    quantile_score = mean_pinball_loss(
            y_val,
            pred_y,
            alpha=0.987531903035467
        )
    huber_score = huber_loss(y_val, pred_y, delta=2.0)
    quantile_scores.append(quantile_score)
    huber_scores.append(huber_score)

    print(f"Fold {fold + 1}: MAE = {mae_score}, RMSE = {rmse_score}, Quantile = {quantile_score}, Huber = {huber_score}")

Fold 1: MAE = 3.542928095738767, RMSE = 12.206906132875853, Quantile = 1.7654436194948853, Huber = 5.46069104493149
Fold 2: MAE = 3.5649833322937665, RMSE = 16.550311370188158, Quantile = 1.7905870422689154, Huber = 5.505906082893866
Fold 3: MAE = 3.5527815116160877, RMSE = 15.220283867409298, Quantile = 1.7850698323252159, Huber = 5.482719935262949
Fold 4: MAE = 3.557251647971673, RMSE = 14.930936933909681, Quantile = 1.7773382595169074, Huber = 5.489598162764324
Fold 5: MAE = 3.5375361786213775, RMSE = 13.56451462614722, Quantile = 1.7593235913356444, Huber = 5.450831712655661


In [13]:
import numpy as np
print(f"Mean MAE = {np.array(mae_scores).mean():.4f}, Mean RMSE = {np.array(rmse_scores).mean():.4f}, \
      Mean Quantile = {np.array(quantile_scores).mean():.4f}, Mean Huber = {np.array(huber_scores).mean():.4f}")

Mean MAE = 3.5511, Mean RMSE = 14.4946,       Mean Quantile = 1.7756, Mean Huber = 5.4779
